In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


sns.set_theme(style="whitegrid")

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)


In [ ]:
# قراءة عينة من الجداول النظيفة
df_demo = pd.read_sql_query("SELECT * FROM demo_clean LIMIT 5", conn)
df_drug = pd.read_sql_query("SELECT * FROM drug_clean LIMIT 5", conn)

display("--- DEMO Clean Sample ---")
display(df_demo.head())

display("--- DRUG Clean Sample ---")
display(df_drug.head())

In [ ]:
# التأكد من توزيع البيانات بين التدريب (الماضي) والاختبار (المستقبل)
query = """
    SELECT is_test_set, COUNT(*) as record_count 
    FROM demo_clean 
    GROUP BY is_test_set
"""
df_split_check = pd.read_sql_query(query, conn)

# 0 = Train (Q1, Q2, Q3) | 1 = Test (Q4)
display(df_split_check)

In [ ]:
# حساب عدد الأدوية لكل مريض (caseid)
query_poly = """
    SELECT caseid, COUNT(final_drug_name) as num_drugs 
    FROM drug_clean 
    GROUP BY caseid
"""
df_poly = pd.read_sql_query(query_poly, conn)

plt.figure(figsize=(12, 6))
sns.countplot(data=df_poly[df_poly['num_drugs'] <= 15], x='num_drugs', palette='viridis')
plt.title('Polypharmacy: Number of Drugs per Patient', weight='bold', fontsize=14)
plt.xlabel('Number of Drugs Taken Concurrently')
plt.ylabel('Number of Patients')
plt.show()

print(f"📌 Average drugs per patient: {df_poly['num_drugs'].mean():.2f}")
print(f"📌 Max drugs for a single patient: {df_poly['num_drugs'].max()}")

In [ ]:
# تصنيف مخرجات المرضى لـ Severe و Non-Severe
query_target = """
    SELECT 
        CASE WHEN outc_cod IN ('DE', 'LT', 'HO') THEN 1 ELSE 0 END as is_severe
    FROM outc_clean
    WHERE outc_cod IS NOT NULL
"""
df_target = pd.read_sql_query(query_target, conn)

target_counts = df_target['is_severe'].value_counts()
labels = ['Non-Severe (0)', 'Severe (1)']

plt.figure(figsize=(8, 8))
plt.pie(target_counts, labels=labels, autopct='%1.2f%%', startangle=90, 
        colors=['#2ECC71', '#E74C3C'], explode=[0, 0.1], shadow=True, textprops={'fontsize': 12, 'weight': 'bold'})
plt.title('Target Variable Distribution (Imbalance Check)', weight='bold', fontsize=15)
plt.show()

scale_pos_weight = target_counts[0] / target_counts[1]
print(f"⚖️ Recommended 'scale_pos_weight' for XGBoost: {scale_pos_weight:.2f}")

In [ ]:
# 1. جلب بيانات العمر ومؤشر التقسيم الزمني من الداتابيز
query_drift = "SELECT age, is_test_set FROM demo_clean WHERE age IS NOT NULL"
df_drift = pd.read_sql_query(query_drift, conn)

# 2. رسم المقارنة (Data Drift Check)
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_drift, x='age', hue='is_test_set', common_norm=False, 
            fill=True, palette=['#3498DB', '#E67E22'], alpha=0.5)

plt.title('Temporal Stability Check: Age Distribution (Train vs. Test)', weight='bold')
plt.xlabel('Age (Years)')

# إضافة الـ Legend (ملاحظة: 0 للتدريب، 1 للاختبار)
plt.legend(title='Dataset', labels=['Test (Q4)', 'Train (Q1-Q3)'])
plt.show()

# 3. تفريغ الميموري
del df_drift

In [ ]:
# التوزيع المتقاطع للعمر والنوع
query_demo_ml = "SELECT age, sex FROM demo_clean WHERE age IS NOT NULL AND sex IN ('M', 'F')"
df_demo_ml = pd.read_sql_query(query_demo_ml, conn)

plt.figure(figsize=(10, 6))
sns.histplot(data=df_demo_ml, x='age', hue='sex', bins=50, kde=True, palette={'M': '#3498DB', 'F': '#E74C3C'})
plt.title('Patient Age Distribution by Gender', weight='bold', fontsize=14)
plt.xlabel('Age (Years)')
plt.ylabel('Frequency')
plt.show()

del df_demo_ml

In [ ]:
# تصنيف مخرجات المرضى (Severe = 1, Non-Severe = 0)
query_target = """
    SELECT 
        CASE WHEN outc_cod IN ('DE', 'LT', 'HO') THEN 1 ELSE 0 END as is_severe
    FROM outc_clean
    WHERE outc_cod IS NOT NULL
"""
df_target = pd.read_sql_query(query_target, conn)

target_counts = df_target['is_severe'].value_counts()
labels = ['Non-Severe (0)', 'Severe (1)']

plt.figure(figsize=(7, 7))
plt.pie(target_counts, labels=labels, autopct='%1.2f%%', startangle=90, 
        colors=['#2ECC71', '#E74C3C'], explode=[0, 0.1], shadow=True)
plt.title('Target Variable Distribution (Imbalance Check)', weight='bold')
plt.show()

scale_weight = target_counts[0] / target_counts[1]
print(f"⚖️ Recommended 'scale_pos_weight' for XGBoost: {scale_weight:.2f}")

del df_target

In [ ]:
# حساب عدد الأدوية لكل مريض (caseid)
query_poly = """
    SELECT caseid, COUNT(final_drug_name) as num_drugs 
    FROM drug_clean 
    GROUP BY caseid
"""
df_poly = pd.read_sql_query(query_poly, conn)

plt.figure(figsize=(10, 5))
# هنفلتر لحد 15 دواء عشان الرسمة تكون واضحة
sns.countplot(data=df_poly[df_poly['num_drugs'] <= 15], x='num_drugs', palette='viridis')
plt.title('Polypharmacy: Number of Drugs per Patient', weight='bold')
plt.xlabel('Number of Concurrent Drugs')
plt.ylabel('Patient Count')
plt.show()

print(f"📌 Average drugs per patient: {df_poly['num_drugs'].mean():.2f}")
print(f"📌 Max drugs for a single patient: {df_poly['num_drugs'].max()}")

del df_poly

In [ ]:
# مقارنة توزيع الأعمار بين التدريب والاختبار
query_drift = "SELECT age, is_test_set FROM demo_clean WHERE age IS NOT NULL"
df_drift = pd.read_sql_query(query_drift, conn)

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_drift, x='age', hue='is_test_set', common_norm=False, 
            fill=True, palette=['#3498DB', '#E67E22'], alpha=0.5)
plt.title('Temporal Stability Check: Age Distribution (Train vs. Test)', weight='bold')
plt.xlabel('Age')
plt.legend(title='Dataset', labels=['Test (Q4)', 'Train (Q1-Q3)'])
plt.show()

del df_drift